# Лабораторная работа 3.4. Численное дифференцирование

**Вариант 7.** По таблице

| $i$ | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| $x_i$ | −0.2 | 0.0 | 0.2 | 0.4 | 0.6 |
| $y_i$ | 1.7722 | 1.5708 | 1.3694 | 1.1593 | 0.9273 |

вычислить первую и вторую производные в точке $X^*=0.2$.

In [1]:
import json
import math
import os

import numpy as np
import pandas as pd

DATA_DIR = "data"

### Вспомогательные функции: чтение данных и оформление вывода

In [2]:
def read_data(name: str) -> dict:
    """Читает JSON-файл с исходными данными."""
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"файл '{path}' не найден: положите папку data рядом с ноутбуком")
    with open(path, encoding="utf-8") as file:
        return json.load(file)


def vector(data: dict, key: str, length: int | None = None) -> np.ndarray:
    """Читает вектор вещественных чисел по ключу и проверяет его длину."""
    if key not in data:
        raise ValueError(f"в исходных данных нет ключа '{key}'")
    values = np.array([float(value) for value in data[key]])
    if length is not None and len(values) != length:
        raise ValueError(f"'{key}': {len(values)} элементов вместо {length}")
    return values


def scalar(data: dict, key: str) -> float:
    """Читает одно вещественное число по ключу."""
    if key not in data:
        raise ValueError(f"в исходных данных нет ключа '{key}'")
    return float(data[key])


def number(value: float, precision: int = 6) -> str:
    """Число в едином формате; машинный нуль печатается как 0."""
    value = float(value)
    if abs(value) < 0.5 * 10 ** (-precision):
        value = 0.0
    return f"{value:.{precision}f}"


def table(rows: list[list], headers: list[str], title: str | None = None) -> None:
    """Печатает таблицу (pandas используется только для оформления)."""
    if title:
        print(title)
    display(pd.DataFrame(rows, columns=headers).style.hide(axis="index"))

### Реализация метода

In [3]:
def choose_nodes(x: np.ndarray, point: float) -> int:
    """Индекс первого из трёх узлов, наиболее симметрично окружающих точку."""
    if point < x[0] or point > x[-1]:
        raise ValueError(f"точка {point} вне таблицы [{x[0]}, {x[-1]}]")
    if len(x) < 3:
        raise ValueError("для вычисления второй производной нужно не менее трёх узлов")
    nearest = int(np.argmin(np.abs(x - point)))
    return min(max(nearest - 1, 0), len(x) - 3)


def slope(x: np.ndarray, y: np.ndarray, first: int, second: int) -> float:
    """Разделённая разность первого порядка (наклон секущей)."""
    return float((y[second] - y[first]) / (x[second] - x[first]))


def first_derivative(x: np.ndarray, y: np.ndarray, i: int, point: float) -> float:
    """Первая производная со вторым порядком точности по узлам x_i, x_(i+1), x_(i+2)."""
    left = slope(x, y, i, i + 1)
    right = slope(x, y, i + 1, i + 2)
    return float(left + (right - left) / (x[i + 2] - x[i]) * (2.0 * point - x[i] - x[i + 1]))


def second_derivative(x: np.ndarray, y: np.ndarray, i: int) -> float:
    """Вторая производная по тем же трём узлам (на участке она постоянна)."""
    left = slope(x, y, i, i + 1)
    right = slope(x, y, i + 1, i + 2)
    return float(2.0 * (right - left) / (x[i + 2] - x[i]))


def bracket(value: float) -> str:
    """Число для подстановки в формулу: отрицательное берётся в скобки."""
    return f"({value:.9f})" if value < 0 else f"{value:.9f}"

### Исходная таблица и точка вычисления (пункты 1 и 2)

In [4]:
data = read_data("lab3_4.json")
x = vector(data, "x")
y = vector(data, "y", len(x))
x_star = scalar(data, "x_star")

table([[i, number(x[i]), number(y[i]), "<- X*" if abs(x[i] - x_star) < 1e-12 else ""]
       for i in range(len(x))],
      ["i", "x_i", "y_i", ""], "1) Исходная таблица значений функции")

print(f"2) Точка вычисления: X* = {x_star}")
if np.min(np.abs(x - x_star)) < 1e-12:
    print(f"   Точка совпадает с узлом таблицы номер {int(np.argmin(np.abs(x - x_star)))}.")
else:
    print("   Точка лежит между узлами таблицы.")

1) Исходная таблица значений функции


i,x_i,y_i,
0,-0.200000,1.772200,
1,0.000000,1.570800,
2,0.200000,1.369400,<- X*
3,0.400000,1.159300,
4,0.600000,0.927300,


2) Точка вычисления: X* = 0.2
   Точка совпадает с узлом таблицы номер 2.


### Узлы, использованные при вычислении (пункт 3)

In [5]:
i = choose_nodes(x, x_star)
print("3) Производные берутся от интерполяционного многочлена второй степени,")
print("   построенного на трёх узлах, наиболее симметрично окружающих X*:")
table([[i + k, number(x[i + k]), number(y[i + k])] for k in range(3)], ["i", "x_i", "y_i"])

3) Производные берутся от интерполяционного многочлена второй степени,
   построенного на трёх узлах, наиболее симметрично окружающих X*:


i,x_i,y_i
1,0.000000,1.570800
2,0.200000,1.369400
3,0.400000,1.159300


### Первая и вторая производные (пункты 4 и 5)

In [6]:
left = slope(x, y, i, i + 1)
right = slope(x, y, i + 1, i + 2)
accurate = first_derivative(x, y, i, x_star)

print("4) Первая производная")
print(f"   Левая разность по узлам x_{i}, x_{i + 1}:   {left:.9f}   (первый порядок точности)")
print(f"   Правая разность по узлам x_{i + 1}, x_{i + 2}: {right:.9f}   (первый порядок точности)")
print("   Уточнение по трём узлам (второй порядок точности):")
print(f"     y'(X*) = {bracket(left)} + ({bracket(right)} - {bracket(left)}) / ({x[i + 2]:g} - {x[i]:g}) * (2*{x_star:g} - {x[i]:g} - {x[i + 1]:g})")
print(f"     y'(X*) = {accurate:.9f}")

print("\n5) Вторая производная")
print(f"   y''(X*) = 2 * ({bracket(right)} - {bracket(left)}) / ({x[i + 2]:g} - {x[i]:g}) = {second_derivative(x, y, i):.9f}")
print("   На участке из трёх узлов вторая производная многочлена второй степени постоянна.")

4) Первая производная
   Левая разность по узлам x_1, x_2:   -1.007000000   (первый порядок точности)
   Правая разность по узлам x_2, x_3: -1.050500000   (первый порядок точности)
   Уточнение по трём узлам (второй порядок точности):
     y'(X*) = (-1.007000000) + ((-1.050500000) - (-1.007000000)) / (0.4 - 0) * (2*0.2 - 0 - 0.2)
     y'(X*) = -1.028750000

5) Вторая производная
   y''(X*) = 2 * ((-1.050500000) - (-1.007000000)) / (0.4 - 0) = -0.217500000
   На участке из трёх узлов вторая производная многочлена второй степени постоянна.
